# instalamos dependencias

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
import subprocess
import sys
import os
def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
print(" Installing latest Transformers")
install("git+https://github.com/huggingface/transformers.git")
pkgs = [
    "accelerate",
    "gradio",
    "numpy",
    "pillow",
    "torch",
    "torchvision",
    "imageio[ffmpeg]"
]
for p in pkgs:
    install(p)

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
import numpy as np
import imageio
from PIL import Image, ImageFilter
from transformers import Sam3Processor, Sam3Model
import time
import cv2

# checando disponibilidad de GPU

In [ ]:
print(f"CUDA Available?: {torch.cuda.is_available()}")
print(f"Num of GPUs availables: {torch.cuda.device_count()}")

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

# cargando sam3 y prediccion de mascaras

In [ ]:
class Sam3PrivacyEngine:
  def __init__(self):
    self.device = "cuda" if torch.cuda.is_available() else "cpu"
    self.model = None
    self.processor = None
      
    print(f"Initializing SAM3 on {self.device}")
    try:
      model_base = Sam3Model.from_pretrained("facebook/sam3").to(self.device)
      self.processor = Sam3Processor.from_pretrained("facebook/sam3")

      if torch.cuda.device_count() > 1:
          print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
          self.model = nn.DataParallel(model_base)
          self.model.to(self.device)
      else:
          self.model = model_base.to(self.device)
        
      print("SAM3 Loaded Successfully")
    except Exception as e:
      print(f"Error Loaded SAM3: {e}")

  def predict_masks_batch(self, image_pil_list, text_prompt_list, threshold=0.4):
      if self.model is None: return []

      inputs = self.processor(
          images=image_pil_list,
          text=text_prompt_list,
          return_tensors="pt"
      ).to(self.device)

      with torch.no_grad():
        outputs = self.model(**inputs)

      target_sizes = inputs["original_sizes"].cpu().tolist()
      batch_results = self.processor.post_process_instance_segmentation(
          outputs=outputs,
          threshold=threshold,
          mask_threshold=0.5,
          target_sizes=target_sizes
      )

      batch_masks = []

      for results in batch_results:
          frame_masks = []
          if "masks" in results:
            for mask_tensor in results["masks"]:
              mask_np = (mask_tensor.cpu().numpy() * 255).astype(np.uint8)
              mask_pil = Image.fromarray(mask_np)
              frame_masks.append(mask_pil)
          batch_masks.append(frame_masks)

      return batch_masks

gine = Sam3PrivacyEngine()

# desenfoque gaussiando

In [ ]:
def apply_blur_pure(image_pil, masks, blur_strength):
  if not masks:
    return image_pil

  radius = blur_strength
  blurred_image = image_pil.filter(ImageFilter.GaussianBlur(radius=radius))
  composite_mask = Image.new("L", image_pil.size, 0)

  for mask in masks:
    composite_mask.paste(255, (0,0), mask=mask)

  final_image = image_pil.copy()
  final_image.paste(blurred_image, (0,0), mask=composite_mask)
  return final_image

# pantalla verde

In [ ]:
def apply_green_screen(image_pil, masks, bg_color=(0,255,0)):
    if not masks:
        return image_pil
    composite_mask = Image.new("L", image_pil.size, 0)
    for mask in masks:
        composite_mask.paste(255, (0,0), mask=mask)

    result = Image.new("RGB", image_pil.size, bg_color)
    result.paste(image_pil, (0,0), mask=composite_mask)
    return result

In [ ]:
def apply_silhouette(image_pil, masks, color=(128,0,128), outline_thickness=3, tint_alpha=0.35):
    if not masks:
        return image_pil
        
    composite_mask = Image.new("L", image_pil.size, 0)
    for mask in masks:
        composite_mask.paste(255, (0,0), mask=mask)
    mask_np = np.array(composite_mask)

    kernel = np.ones((outline_thickness * 2 + 1, outline_thickness * 2 + 1), np.uint8)
    dilated = cv2.dilate(mask_np, kernel, iterations=1)
    outline_np = (dilated - mask_np).clip(0, 255).astype(np.uint8)

    result = np.array(image_pil, dtype=np.float32)

    tint = np.array(color, dtype=np.float32)
    subject_region = mask_np > 0
    result[subject_region] = (
        result[subject_region] * (1 - tint_alpha) + tint * tint_alpha
    )
    
    outline_region = outline_np > 0
    result[outline_region] = tint
    return Image.fromarray(result.clip(0,255).astype(np.uint8))

In [ ]:
def apply_pixelate(image_pil, masks, block_size=20):
    if not masks:
        return image_pil
    composite_mask = Image.new("L", image_pil.size, 0)
    for mask in masks:
        composite_mask.paste(255, (0,0), mask=mask)
    w, h = image_pil.size
    pixelated = image_pil.resize((w // block_size, h // block_size), Image.NEAREST)
    pixelated = pixelated.resize((w, h), Image.NEAREST)
    result = image_pil.copy()
    result.paste(pixelated, (0,0), mask=composite_mask)
    return result

## proceso de imagenes

In [ ]:
def process_image(input_img, text_prompt, blur_strenght, confidence):
  if input_img is None: return None, None
  if isinstance(input_img, np.ndarray):
    image_pil = Image.fromarray(input_img).convert("RGB")
  else:
    image_pil = input_img.convert("RGB")

  masks = gine.predict_masks(image_pil, text_prompt, confidence)
  result_pil = apply_blur_pure(image_pil, masks, blur_strenght)
  output_path = "privacy_image.png"
  result_pil.save(output_path)
  return np.array(result_pil), output_path

## canal de video

In [ ]:
EFFECTS = {
    "blur":             apply_blur_pure,
    "green_screen":     apply_green_screen,
    "silhouette":       apply_silhouette,
    "pixelate":         apply_pixelate,
}

In [ ]:
def process_video(video_path, text_prompt, effect="blur", effect_params=None, confidence=0.3, max_frames=150, batch_size=8):
  if not video_path: return None, None

  effect_fn = EFFECTS.get(effect)
  if effect_fn is None:
      print(f"There is no '{effect}', please enter a valid effect: {list(EFFECTS.keys())}")
      return None, None

  effect_params = effect_params or {}
    
  try:
    reader = imageio.get_reader(video_path)
    meta = reader.get_meta_data()
    fps = meta.get('fps', 24)
    output_path = f"/kaggle/working/privacy_video_{effect}.mp4"
    writer = imageio.get_writer(
        output_path,
        fps=fps,
        codec='libx264',
        pixelformat='yuv420p',
        macro_block_size=1
    )

    print(f"Starting video processing '{effect}'...")

    frame_buffer = []
    def process_and_write_batch(buffer):
        prompts_list = [text_prompt] * len(buffer)
        batch_masks = gine.predict_masks_batch(buffer, prompts_list, confidence)

        for frame_pil, masks in zip(buffer, batch_masks):
            processed_pil = effect_fn(frame_pil, masks, **effect_params)
            writer.append_data(np.array(processed_pil))
      
    for i, frame in enumerate(reader):
      if i >= max_frames:
        break

      frame_pil = Image.fromarray(frame).convert("RGB")
      frame_buffer.append(frame_pil)

      if len(frame_buffer) == batch_size:
          process_and_write_batch(frame_buffer)
          frame_buffer = []

      if i % 10 == 0:
          print(f"Frame {i}")


    if frame_buffer:
        process_and_write_batch(frame_buffer)

      
    writer.close()
    reader.close()
    print("video processing complete.")
    return output_path, output_path
  except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()
    return None, None

# visualization

In [ ]:
from IPython.display import Video, display

In [ ]:
INPUT = "/kaggle/input/datasets/gerardomacias/test-video/WIN_20260518_16_19_15_Pro.mp4"

In [ ]:
video = process_video(
    INPUT,
    "face",
    effect="blur",
    effect_params={"blur_strength": 51}
)

if video[0]:
    display(Video(video[0], embed=True, width=640))
else:
    print("El procesamiento falló, no hay video que mostrar.")

In [ ]:
video = process_video(
    INPUT,
    "person",
    effect="green_screen",
    effect_params={"bg_color": (0, 255, 0)}
)

if video[0]:
    display(Video(video[0], embed=True, width=640))
else:
    print("El procesamiento falló, no hay video que mostrar.")

In [ ]:
video = process_video(
    INPUT,
    "person",
    effect="silhouette",
    effect_params={"color": (128, 0, 128), "outline_thickness": 3, "tint_alpha": 0.35}
)

if video[0]:
    display(Video(video[0], embed=True, width=640))
else:
    print("El procesamiento falló, no hay video que mostrar.")

In [ ]:
video = process_video(
    INPUT,
    "person",
    effect="pixelate",
    effect_params={"block_size": 20}
)

if video[0]:
    display(Video(video[0], embed=True, width=640))
else:
    print("El procesamiento falló, no hay video que mostrar.")